## ML Pipeline for Airfoil noise prediction


## Objectivos
- Parte 1 Realizar actividad ETL
- Cargar un conjunto de datos CSV
- Eliminar duplicados si los hay
- Eliminar filas con valores nulos si los hay
- Realizar transformaciones
- Almacenar los datos limpios en formato parquet
- Parte 2 Crear un Pipeline de Aprendizaje Automático
- Crear un pipeline de aprendizaje automático para predicción
- Parte 3 Evaluar el Modelo
- Evaluar el modelo usando métricas relevantes
- Parte 4 Persistir el Modelo
- Guardar el modelo para uso futuro en producción
- Cargar y verificar el modelo almacenado


In [1]:
!pip install pyspark==3.1.2 -q
!pip install findspark -q

In [3]:
# You can also use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

# FindSpark simplifies the process of using Apache Spark with Python

import findspark
findspark.init()

## ETL

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, regexp_replace, to_date

In [ ]:
# SparkSession
spark = SparkSession.builder \
    .appName("ETL_Final_Project") \
    .getOrCreate()

26/01/20 23:58:07 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [7]:
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-BD0231EN-Coursera/datasets/NASA_airfoil_noise_raw.csv


--2026-01-20 23:58:17--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-BD0231EN-Coursera/datasets/NASA_airfoil_noise_raw.csv
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.63.118.104, 169.63.118.104
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.63.118.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 60682 (59K) [text/csv]
Saving to: ‘NASA_airfoil_noise_raw.csv’

NASA_airfoil_noise_ 100%[===================>]  59.26K  --.-KB/s    in 0.002s  

2026-01-20 23:58:18 (37.5 MB/s) - ‘NASA_airfoil_noise_raw.csv’ saved [60682/60682]



In [ ]:
# Cargar df con spark.read.csv, indicando que el archivo tiene header y que se infieran los tipos de datos

df = spark.read.csv("NASA_airfoil_noise_raw.csv", header=True, inferSchema=True)

### EDA

In [ ]:
df.show(5)

+---------+-------------+-----------+------------------+-----------------------+----------+
|Frequency|AngleOfAttack|ChordLength|FreeStreamVelocity|SuctionSideDisplacement|SoundLevel|
+---------+-------------+-----------+------------------+-----------------------+----------+
|      800|          0.0|     0.3048|              71.3|             0.00266337|   126.201|
|     1000|          0.0|     0.3048|              71.3|             0.00266337|   125.201|
|     1250|          0.0|     0.3048|              71.3|             0.00266337|   125.951|
|     1600|          0.0|     0.3048|              71.3|             0.00266337|   127.591|
|     2000|          0.0|     0.3048|              71.3|             0.00266337|   127.461|
+---------+-------------+-----------+------------------+-----------------------+----------+
only showing top 5 rows



In [13]:
#your code goes here
rowcount1 = df.count()
print(rowcount1)

1522


In [ ]:
# Eliminar filas duplicadas
df = df.dropDuplicates()

In [ ]:
# Contar el número de filas después de eliminar duplicados
rowcount2 = df.count()
print(rowcount2)

[Stage 8:=======================================================(200 + 0) / 200]

1503


In [ ]:
# Eliminar filas con valores nulos
df = df.dropna()

In [ ]:
rowcount3 = df.count()
print(rowcount3)


[Stage 11:====================================================> (196 + 4) / 200]

1499


In [ ]:
# Renombrar la columna "SoundLevel" a "SoundLevelDecibels"
df = df.withColumnRenamed("SoundLevel", "SoundLevelDecibels")

In [ ]:
# Convertir df en parquet
df.write.parquet("NASA_airfoil_noise_cleaned.parquet")

[Stage 14:>                                                       (0 + 8) / 200]26/01/21 00:02:55 WARN hadoop.MemoryManager: Total allocation exceeds 95.00% (906,992,014 bytes) of heap memory
Scaling row group sizes to 96.54% for 7 writers
26/01/21 00:02:55 WARN hadoop.MemoryManager: Total allocation exceeds 95.00% (906,992,014 bytes) of heap memory
Scaling row group sizes to 84.47% for 8 writers
26/01/21 00:02:56 WARN hadoop.MemoryManager: Total allocation exceeds 95.00% (906,992,014 bytes) of heap memory
Scaling row group sizes to 96.54% for 7 writers


#### Evalucion

In [23]:
print("Part 1 - Evaluation")

print("Total rows = ", rowcount1)
print("Total rows after dropping duplicate rows = ", rowcount2)
print("Total rows after dropping duplicate rows and rows with null values = ", rowcount3)
print("New column name = ", df.columns[-1])

import os

print("NASA_airfoil_noise_cleaned.parquet exists :", os.path.isdir("NASA_airfoil_noise_cleaned.parquet")

Part 1 - Evaluation
Total rows =  1522
Total rows after dropping duplicate rows =  1503
Total rows after dropping duplicate rows and rows with null values =  1499
New column name =  SoundLevelDecibels
NASA_airfoil_noise_cleaned.parquet exists : True


## Creacion de Machine Learning Pipeline

In [ ]:
# Carga de datos
df = spark.read.parquet("NASA_airfoil_noise_cleaned.parquet")

In [ ]:
rowcount4 = df.count()
print(rowcount4)

[Stage 16:=====================>                                    (3 + 5) / 8]

1499


In [ ]:
from pyspark.ml.feature import VectorAssembler

input_features = [col for col in df.columns if col != "SoundLevelDecibels"]
assembler = VectorAssembler(inputCols=input_features, outputCol="features") # Crear un ensamblador de características para combinar las columnas de entrada en una sola columna de características

In [ ]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withMean=True, withStd=True) # Crear un escalador estándar para normalizar las características

### Creacion del modelo pipeline

In [ ]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol="scaledFeatures", labelCol="SoundLevelDecibels") # Crear un modelo de regresión lineal para predecir el nivel de sonido en decibelios utilizando las características escaladas

### Creacion del pipeline

In [ ]:
#your code goes here

from pyspark.ml import Pipeline

pipeline = Pipeline(stages=[assembler, scaler, lr]) # Crear un pipeline con las etapas de ensamblado, escalado y regresión lineal

In [ ]:
(trainingData, testingData) = df.randomSplit([0.7, 0.3], seed=42)


### Fit pipeline


In [ ]:
pipelineModel = pipeline.fit(trainingData)

26/01/21 00:08:49 WARN util.Instrumentation: [719ade9c] regParam is zero, which might cause numerical instability and overfitting.
[Stage 21:>                                                         (0 + 8) / 8]26/01/21 00:08:51 WARN netlib.BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeSystemBLAS
26/01/21 00:08:51 WARN netlib.BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeRefBLAS
26/01/21 00:08:51 WARN netlib.LAPACK: Failed to load implementation from: com.github.fommil.netlib.NativeSystemLAPACK
26/01/21 00:08:51 WARN netlib.LAPACK: Failed to load implementation from: com.github.fommil.netlib.NativeRefLAPACK


#### Evaluacion

In [34]:
print("Part 2 - Evaluation")
print("Total rows = ", rowcount4)
ps = [str(x).split("_")[0] for x in pipeline.getStages()]

print("Pipeline Stage 1 = ", ps[0])
print("Pipeline Stage 2 = ", ps[1])
print("Pipeline Stage 3 = ", ps[2])

print("Label column = ", lr.getLabelCol())

Part 2 - Evaluation
Total rows =  1499
Pipeline Stage 1 =  VectorAssembler
Pipeline Stage 2 =  StandardScaler
Pipeline Stage 3 =  LinearRegression
Label column =  SoundLevelDecibels


## Evaluacion del Modelo

In [ ]:
predictions = pipelineModel.transform(testingData)

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator_mse = RegressionEvaluator(
    labelCol="SoundLevelDecibels",
    predictionCol="prediction",
    metricName="mse"
)

mse = evaluator_mse.evaluate(predictions)
print(mse)


[Stage 28:>                                                         (0 + 8) / 8]

22.593754071348833


In [ ]:
evaluator_mae = RegressionEvaluator(
    labelCol="SoundLevelDecibels",
    predictionCol="prediction",
    metricName="mae"
)

mae = evaluator_mae.evaluate(predictions)
print(mae)


[Stage 30:>                                                         (0 + 8) / 8]

3.7336902294631447


In [ ]:
evaluator_r2 = RegressionEvaluator(
    labelCol="SoundLevelDecibels",
    predictionCol="prediction",
    metricName="r2"
)

r2 = evaluator_r2.evaluate(predictions)
print(r2)

[Stage 32:>                                                         (0 + 8) / 8]

0.5426016508689053


In [39]:
print("Part 3 - Evaluation")

print("Mean Squared Error = ", round(mse,2))
print("Mean Absolute Error = ", round(mae,2))
print("R Squared = ", round(r2,2))

lrModel = pipelineModel.stages[-1]

print("Intercept = ", round(lrModel.intercept,2))


Part 3 - Evaluation
Mean Squared Error =  22.59
Mean Absolute Error =  3.73
R Squared =  0.54
Intercept =  125.14


## Part 4 - Persist the Model


In [ ]:
from pyspark.ml.pipeline import PipelineModel

loadedPipelineModel = PipelineModel.load("Final_Project") # Cargar el modelo de pipeline que has creado en el paso anterior

In [ ]:
predictions = loadedPipelineModel.transform(testingData) # Use el modelo de pipeline cargado y hacer predicciones usando testingData

In [ ]:
predictions.select("SoundLevelDecibels", "prediction").show(5) # Mostrar las primeras 5 filas del DataFrame de predicciones, mostrando solo la columna de etiqueta y las predicciones


[Stage 54:>                                                         (0 + 1) / 1]

+------------------+------------------+
|SoundLevelDecibels|        prediction|
+------------------+------------------+
|           127.315|  123.643440096247|
|           119.975|123.48695788614808|
|           121.783|124.38983849684215|
|           127.224|121.44706993294244|
|           122.229|125.68312652454135|
+------------------+------------------+
only showing top 5 rows



In [45]:
print("Part 4 - Evaluation")

loadedmodel = loadedPipelineModel.stages[-1]
totalstages = len(loadedPipelineModel.stages)
inputcolumns = loadedPipelineModel.stages[0].getInputCols()

print("Number of stages in the pipeline = ", totalstages)
for i,j in zip(inputcolumns, loadedmodel.coefficients):
    print(f"Coefficient for {i} is {round(j,4)}")

Part 4 - Evaluation
Number of stages in the pipeline =  3
Coefficient for Frequency is -3.9728
Coefficient for AngleOfAttack is -2.4775
Coefficient for ChordLength is -3.3818
Coefficient for FreeStreamVelocity is 1.5789
Coefficient for SuctionSideDisplacement is -1.6465


In [ ]:
spark.stop() # Detener la sesión de Spark

<!--
|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2023-05-26|0.1|Ramesh Sannareddy|Initial Version Created|
-->
